# 2. LLM API Fundamentals & Production Mechanics

This notebook covers interaction patterns with Large Language Model APIs:
1. **Request/Response Schemas** (System, User, Assistant message formatting)
2. **Decoding Parameters Visualizer** (Temperature, Top-P, Top-K, Presence & Frequency Penalties)
3. **Simulated Token Streaming Generator**
4. **Tool / Function Calling Protocol & Loop**
5. **Structured Output Parsing & Schema Enforcer**

In [ ]:
import time
import json
import random
import math
from typing import List, Dict, Any, Generator

print("=== LLM API Setup Complete ===")

## Step 1: Open-AI Style Request & Response Schemas

LLM Chat APIs follow a structured messaging protocol with distinct roles:
- `system`: Sets model instructions, persona, and behavioral boundaries.
- `user`: Incoming prompt or command from the end-user.
- `assistant`: Previous model outputs in the turn sequence.

In [ ]:
class SimulatedLLMClient:
    def __init__(self, model_name: str = "gpt-4o-mock"):
        self.model_name = model_name

    def create_chat_completion(
        self,
        messages: List[Dict[str, str]],
        temperature: float = 0.7,
        top_p: float = 0.9,
        max_tokens: int = 100
    ) -> Dict[str, Any]:
        last_user_msg = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
        
        simulated_response = f"Simulated response to: '{last_user_msg}' [temp={temperature}, top_p={top_p}]"
        
        return {
            "id": f"chatcmpl-{random.randint(10000, 99999)}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": self.model_name,
            "choices": [
                {
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": simulated_response
                    },
                    "finish_reason": "stop"
                }
            ],
            "usage": {
                "prompt_tokens": sum(len(m["content"].split()) for m in messages),
                "completion_tokens": len(simulated_response.split()),
                "total_tokens": sum(len(m["content"].split()) for m in messages) + len(simulated_response.split())
            }
        }

client = SimulatedLLMClient()
conversation = [
    {"role": "system", "content": "You are a helpful AI assistant specialized in API design."},
    {"role": "user", "content": "Explain the difference between REST and gRPC."}
]

res = client.create_chat_completion(conversation, temperature=0.3)
print(json.dumps(res, indent=2))

## Step 2: Decoding Parameters & Sampling Simulator

LLMs predict token probability distributions. Decoding parameters alter this distribution:
- **Temperature ($T$)**: Scales logits ($z_i / T$). Higher values ($T>1.0$) flatten distribution (more creative), lower values ($T\to 0$) make it greedy.
- **Top-P (Nucleus Sampling)**: Keeps smallest set of tokens whose cumulative probability $\ge P$.
- **Top-K**: Keeps top $K$ highest probability tokens.
- **Frequency Penalty**: Penalizes tokens based on their cumulative frequency in generated text.
- **Presence Penalty**: Penalizes tokens if they have appeared at least once in generated text.

In [ ]:
def simulate_token_sampling(
    token_logits: Dict[str, float],
    temperature: float = 1.0,
    top_p: float = 1.0,
    top_k: int = 0
) -> Dict[str, float]:
    # 1. Apply Temperature Scaling
    temp = max(temperature, 1e-5)
    scaled_logits = {t: l / temp for t, l in token_logits.items()}
    
    # Softmax
    max_logit = max(scaled_logits.values())
    exp_logits = {t: math.exp(l - max_logit) for t, l in scaled_logits.items()}
    sum_exp = sum(exp_logits.values())
    probs = {t: e / sum_exp for t, e in exp_logits.items()}
    
    # Sort tokens by probability descending
    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)

    # 2. Top-K Filter
    if top_k > 0 and top_k < len(sorted_probs):
        sorted_probs = sorted_probs[:top_k]

    # 3. Top-P Filter
    cum_prob = 0.0
    filtered_probs = {}
    for token, prob in sorted_probs:
        cum_prob += prob
        filtered_probs[token] = prob
        if cum_prob >= top_p:
            break
            
    # Re-normalize filtered probabilities
    total_f = sum(filtered_probs.values())
    return {t: round(p / total_f, 4) for t, p in filtered_probs.items()}

# Sample raw unnormalized logits
mock_logits = {"API": 4.5, "HTTP": 3.8, "JSON": 3.2, "Banana": 0.5, "Quantum": 0.1}

print("Deterministic Greedy (T=0.1):", simulate_token_sampling(mock_logits, temperature=0.1))
print("Creative Sampling (T=1.5):", simulate_token_sampling(mock_logits, temperature=1.5))
print("Nucleus Sampling (Top-P=0.8):", simulate_token_sampling(mock_logits, temperature=1.0, top_p=0.8))

## Step 3: Simulated Response Streaming

Token streaming emits output chunks as they are generated, reducing Time-To-First-Token (TTFT) for user interfaces.

In [ ]:
def stream_chat_completion(prompt: str) -> Generator[Dict[str, Any], None, None]:
    words = f"Streaming response for query: '{prompt}'. LLMs process text in sequential tokens.".split()
    for idx, word in enumerate(words):
        chunk = {
            "id": "chatcmpl-stream-1",
            "object": "chat.completion.chunk",
            "choices": [
                {
                    "index": 0,
                    "delta": {"content": word + " "},
                    "finish_reason": "stop" if idx == len(words) - 1 else None
                }
            ]
        }
        yield chunk

print("Simulated Token Stream Output:")
for chunk in stream_chat_completion("Demonstrate streaming"):
    token = chunk["choices"][0]["delta"].get("content", "")
    print(token, end="", flush=True)
print("\n[Stream Complete]")

## Step 4: Tool / Function Calling Protocol

Function calling enables models to interact with external tools by generating structured tool call requests instead of plain text.

In [ ]:
def get_current_weather(location: str, unit: str = "celsius") -> str:
    return f"The weather in {location} is 22°{unit.upper()[0]} and sunny."

available_tools = {
    "get_current_weather": get_current_weather
}

tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location"]
            }
        }
    }
]

def simulate_tool_calling_loop(user_query: str):
    print(f"User Query: '{user_query}'")
    # Step 1: Model decides to call a function
    simulated_tool_call = {
        "id": "call_abc123",
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "arguments": json.dumps({"location": "Berlin", "unit": "celsius"})
        }
    }
    print(f"-> Model requested tool call: {simulated_tool_call['function']['name']} with args {simulated_tool_call['function']['arguments']}")

    # Step 2: Execute tool locally
    fn_name = simulated_tool_call["function"]["name"]
    args = json.loads(simulated_tool_call["function"]["arguments"])
    tool_output = available_tools[fn_name](**args)
    print(f"-> Executed Tool Result: '{tool_output}'")

    # Step 3: Send tool result back to LLM for final synthesis
    final_synthesis = f"Based on the live data: {tool_output}"
    print(f"-> Final Assistant Response: '{final_synthesis}'")

simulate_tool_calling_loop("What is the temperature in Berlin?")

## Step 5: Structured Response Parsing & Validation

Ensuring LLMs return strict, valid JSON matching expected data schemas is essential for downstream systems.

In [ ]:
class StructuredOutputEnforcer:
    @staticmethod
    def parse_and_validate(raw_llm_json: str, required_keys: List[str]) -> Tuple[bool, Dict[str, Any], str]:
        try:
            data = json.loads(raw_llm_json)
            for key in required_keys:
                if key not in data:
                    return False, {}, f"Missing required key: '{key}'"
            return True, data, "Validation successful"
        except json.JSONDecodeError as e:
            return False, {}, f"Invalid JSON syntax: {str(e)}"

# Test 1: Valid JSON response
valid_response = '{"city": "Munich", "population": 1500000, "is_capital": false}'
success, parsed, msg = StructuredOutputEnforcer.parse_and_validate(valid_response, ["city", "population"])
print(f"Test 1 Valid JSON: Success={success}, Data={parsed}")

# Test 2: Malformed response handling
malformed_response = '{"city": "Hamburg", "population": 1800000,' # trailing comma / unclosed
success, parsed, msg = StructuredOutputEnforcer.parse_and_validate(malformed_response, ["city", "population"])
print(f"Test 2 Malformed JSON: Success={success}, Error='{msg}'")